# Step 4: LightGBM hybrid model

The main model: a gradient-boosted tree ranker that scores every (user, candidate) pair from four groups of engineered features (feature schema contract v0.1), then ranks candidates within each impression.

| Group | Inputs | Role |
|---|---|---|
| T1, article behaviour | prior exposures and clicks, prior CTR, evidence weight, 6-hour exposures/clicks, CTR momentum, age | What the candidate's own history says |
| T2, content-transferred behaviour | recent CTR and momentum of the same category, subcategory and entities; semantic-neighbour CTR | Behaviour borrowed from related articles, available even when the candidate has none |
| T3, pure content | category and subcategory codes, entity count and coverage, 32 title PCs, 16 entity PCs | What the article is about |
| U, user–article matching | history length, category/subcategory affinity, text and entity similarity, entity overlap, user profile PCs | How well the candidate fits this user; kept in every ablation |

```
meta + features (row tables) ─┐
article_vectors ── PCA (fit on training-accessible articles) ─┼─ join by news_id ──┐
user_vectors ───── same PCA ──────────────────────────────────┴─ join by impression ─┼─► LightGBM ─► score ─► rank within impression
                                                                                     │
                                                  early stopping on validation ◄────┘
```

**Objective.** Binary log-loss, so scores are click probabilities, with early stopping on validation AUC. LambdaRank, which optimises nDCG directly within impressions, is trained afterwards with the same settings as a comparison.

**No negative sampling.** All candidates are kept for training. At 1 click per 24 non-clicks this fits comfortably in free-Colab memory, and the trees see the real class balance. (The interim report anticipated 1:4 sampling; this is a deliberate change.)

**Tuning.** Four tree-shape settings are trained on the train split and compared on validation only. Selection rule: highest validation Cold Recall@10 among settings within 0.005 validation nDCG@10 of the best. The tolerance is provisional until the team agrees it. The test split is scored once, after selection.

**Memory.** Embedding vectors stay in their own tables. Each article and user is reduced once with PCA, cached, and joined onto the rows as float32 columns.

> Metrics here come from `shared/eval_interim.py`, a stand-in used for tuning until the team evaluator exists. The numbers in the final report come from `step6_evaluation_diagnostics.ipynb`.

## 1. Setup

In [ ]:
import os, sys, subprocess, json, time

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Bryng23/BT4222-Group-11.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/BT4222-Group-11"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
    DATA_ROOT = "/content/drive/MyDrive/BT4222_G11"
else:
    REPO_DIR = os.getcwd()
    DATA_ROOT = os.path.join(REPO_DIR, "data")
sys.path.insert(0, REPO_DIR)

import numpy as np
import pandas as pd
from shared import schema as S, feature_groups as FG, lgbm_pipeline as L, selection as SEL
from shared.predictions import write_predictions
from shared import eval_interim as E      # swap for the team evaluator once it exists

DATA_KIND = os.environ.get("G11_DATA_KIND", "REAL")   # SYNTHETIC only for local development
if DATA_KIND == "REAL":
    import kagglehub
    RAW_DIR = os.path.join(kagglehub.dataset_download("arashnic/mind-news-dataset"), "MINDsmall_train")
    FEATURE_ROOT = os.path.join(DATA_ROOT, "features", f"v{S.SCHEMA_VERSION}")
else:
    FEATURE_ROOT = os.path.join(DATA_ROOT, "features", f"synthetic_v{S.SCHEMA_VERSION}", "learnable")
    RAW_DIR = os.path.join(FEATURE_ROOT, "raw")
version = S.read_version(FEATURE_ROOT)
assert version.get("data_kind", "REAL") == DATA_KIND, f"feature tables are {version.get('data_kind')}, run is {DATA_KIND}"
PRED_ROOT = os.path.join(DATA_ROOT, "predictions")
MODEL_DIR = os.path.join(DATA_ROOT, "models", DATA_KIND.lower())
os.makedirs(MODEL_DIR, exist_ok=True)
SEED = 4222
N_TITLE_PC, N_ENT_PC = 32, 16
INCLUDE_USER_PC = True      # user profile vectors projected with the article PCA; set False if RAM runs short
SHOW = ["auc", "mrr", "ndcg@5", "ndcg@10", "cold_recall@10", "cold_hit@10", "warm_recall@10"]
print(f"DATA_KIND={DATA_KIND} | features: {FEATURE_ROOT} | schema {version['schema_version']}")

## 2. Reduce vectors and load the row tables

PCA sees only articles available during training: training candidates and every article in the click histories of training impressions. Validation and test articles are projected with the frozen components.

In [ ]:
t0 = time.time()
train_meta_ids = pd.read_parquet(S.meta_path(FEATURE_ROOT, "train"), columns=["impression_id", "news_id"])
accessible = set(train_meta_ids["news_id"].astype(str)) | L.train_histories(RAW_DIR, train_meta_ids["impression_id"].unique())
reducer = L.VectorReducer(N_TITLE_PC, N_ENT_PC, seed=SEED).fit(FEATURE_ROOT, accessible)
print(f"PCA fitted on {reducer.n_fit_articles:,} training-accessible articles "
      f"({reducer.n_fit_entity_articles:,} with entity vectors); explained variance {reducer.explained_variance()}")

data = {split: L.load_split(FEATURE_ROOT, split, reducer, include_user_pc=INCLUDE_USER_PC) for split in S.SPLITS}
INCLUDE_SEMANTIC = "semantic_neighbor_ctr" in data["train"][1].columns   # optional in the contract
GROUPS = FG.group_columns(N_TITLE_PC, N_ENT_PC, include_user_pc=INCLUDE_USER_PC, include_semantic=INCLUDE_SEMANTIC)
print("semantic_neighbor_ctr present:", INCLUDE_SEMANTIC)
pd.DataFrame({split: {"rows": len(m), "impressions": m["impression_id"].nunique(), "columns available": x.shape[1],
                      "memory MB": round(x.memory_usage(deep=True).sum() / 1e6, 1)}
              for split, (m, x) in data.items()}).T

## 3. Tune tree shape on validation

In [ ]:
FULL_COLS = FG.feature_columns("full", n_title_pc=N_TITLE_PC, n_ent_pc=N_ENT_PC, include_user_pc=INCLUDE_USER_PC,
                         include_semantic=INCLUDE_SEMANTIC)
(m_tr, x_tr), (m_va, x_va) = data["train"], data["val"]
X_tr, X_va = L.design(x_tr, FULL_COLS), L.design(x_va, FULL_COLS)
print(f"full model: {len(FULL_COLS)} columns")

boosters, rows = {}, []
for params in L.PARAM_GRID:
    pid = L.param_id(params)
    t0 = time.time()
    b = L.fit(X_tr, m_tr["label"], X_va, m_va["label"], params, seed=SEED)
    boosters[pid] = b
    res = E.evaluate(m_va, L.predict(b, X_va))
    rows.append({"params": pid, "best_iteration": b.best_iteration, "minutes": round((time.time() - t0) / 60, 1),
                 **{k: res[k] for k in SHOW}})
grid = pd.DataFrame(rows).set_index("params")
SELECTED = SEL.select(grid)
selected_params = next(p for p in L.PARAM_GRID if L.param_id(p) == SELECTED)
print(f"selected {SELECTED}: {SEL.describe()}")
with open(os.path.join(MODEL_DIR, "lgbm_selected_params.json"), "w") as f:
    json.dump({"params": selected_params, "best_iteration": boosters[SELECTED].best_iteration,
               "n_title_pc": N_TITLE_PC, "n_ent_pc": N_ENT_PC, "include_user_pc": INCLUDE_USER_PC}, f, indent=1)
grid.round(4)

## 4. Score every split with the selected model

Train, validation and test are scored on full slates and written as prediction files. This is the only time the test split is scored.

In [ ]:
full = boosters[SELECTED]
final = {}
for split, (m, x) in data.items():
    sc = L.predict(full, L.design(x, FULL_COLS))
    write_predictions(m["row_id"], sc, root=PRED_ROOT, data_kind=DATA_KIND, model_id="lgbm_full", config_id=SELECTED,
                      split=split, schema_version=S.SCHEMA_VERSION,
                      extra={"objective": "binary", "best_iteration": full.best_iteration, "selected_on": "val"})
    final[split] = E.evaluate(m, sc)
pd.DataFrame(final).T[SHOW + ["n_cold_click_impressions"]].round(4)

## 5. Comparison with the step 3 baselines

Reads whichever baseline prediction files exist for this data kind and evaluates them on the same rows with the same metric code.

In [ ]:
import glob
from shared.predictions import read_predictions
cmp_rows = []
for split in ("val", "test"):
    m = data[split][0]
    cmp_rows.append({"split": split, "model": "lgbm_full", **{k: final[split][k] for k in SHOW}})
    for path in sorted(glob.glob(os.path.join(PRED_ROOT, DATA_KIND.lower(), f"{DATA_KIND}__*__{split}.parquet"))):
        p, info = read_predictions(path)
        if info.get("selected_on") != "val" or info["model_id"].startswith("lgbm"):
            continue   # only the frozen baseline choices
        sc = m[["row_id"]].merge(p, on="row_id", how="left")["predicted_score"]
        if sc.isna().any():
            continue
        cmp_rows.append({"split": split, "model": info["model_id"], **{k: E.evaluate(m, sc.to_numpy())[k] for k in SHOW}})
pd.DataFrame(cmp_rows).set_index(["split", "model"]).round(4)

## 6. Which feature groups the model leans on

Share of total split gain by group and the top individual features. Gain shows what the trees used, not what each group adds on its own; the ablation study in step 5 measures that.

In [ ]:
imp = L.importance_by_group(full, GROUPS)
from IPython.display import display
display(imp.groupby("group")["gain_share"].sum().sort_values(ascending=False).round(4).to_frame())
imp.head(20).round(4)

## 7. LambdaRank comparison

Same columns and tree shape, but optimising nDCG within each impression (rows grouped by impression) with early stopping on validation nDCG@10. Its scores are ranking scores, not probabilities.

In [ ]:
g_tr, g_va = L.impression_groups(m_tr), L.impression_groups(m_va)
rank = L.fit(X_tr, m_tr["label"], X_va, m_va["label"], {**selected_params, "objective": "lambdarank"},
             seed=SEED, group_tr=g_tr, group_va=g_va)
lr_rows = {}
for split, (m, x) in data.items():
    sc = L.predict(rank, L.design(x, FULL_COLS))
    write_predictions(m["row_id"], sc, root=PRED_ROOT, data_kind=DATA_KIND, model_id="lgbm_full_lambdarank",
                      config_id=SELECTED, split=split, schema_version=S.SCHEMA_VERSION,
                      extra={"objective": "lambdarank", "best_iteration": rank.best_iteration, "selected_on": "val"})
    lr_rows[split] = E.evaluate(m, sc)
pd.concat({"binary": pd.DataFrame(final).T[SHOW], "lambdarank": pd.DataFrame(lr_rows).T[SHOW]}).round(4)

## 8. Reading the results

* The baseline comparison in section 5 is the headline table: does the hybrid model beat `category_ctr`, the smoothed-CTR baselines and Group 10's model overall, and on Cold Recall@10?
* Group gain shares in section 6 say where the trees found signal. The ablations in step 5 say what each group is worth.
* LambdaRank is reported for completeness. The binary model stays the main model unless LambdaRank wins on validation under the same selection rule.